In [4]:
# STEP 1 — IMPORT LIBRARIES
import pandas as pd
import numpy as np
import re
import string
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding,
    Bidirectional,
    LSTM,
    Dense,
    Dropout
)
print("Libraries Loaded Successfully")

Libraries Loaded Successfully


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
# STEP 2 — LOAD DATASET
df = pd.read_csv("/content/drive/MyDrive/WELFake_Dataset[2].csv")
df.head()

,Unnamed: 0,title,text,label
0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1
1,1,NaN,Did they post their votes for Hillary already?,1
2,2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1
3,3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0
4,4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1


In [7]:
# STEP 3 — EXPLORE DATASET
# Check dataset size and class distribution
print(df.head())

print(df.shape)

print(df['label'].value_counts())

   Unnamed: 0                                              title  \
0           0  LAW ENFORCEMENT ON HIGH ALERT Following Threat...   
1           1                                                NaN   
2           2  UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...   
3           3  Bobby Jindal, raised Hindu, uses story of Chri...   
4           4  SATAN 2: Russia unvelis an image of its terrif...   

                                                text  label  
0  No comment is expected from Barack Obama Membe...      1  
1     Did they post their votes for Hillary already?      1  
2   Now, most of the demonstrators gathered last ...      1  
3  A dozen politically active pastors came here f...      0  
4  The RS-28 Sarmat missile, dubbed Satan 2, will...      1  
(72134, 4)
label
1    37106
0    35028
Name: count, dtype: int64


In [8]:
# STEP 4 — KEEP ONLY REQUIRED COLUMNS
# Remove unnecessary columns
df = df[['text', 'label']]

df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)

print(df.shape)

(72095, 2)


In [9]:
# STEP 5 — TEXT CLEANING FUNCTION
# Lowercase, remove URLs, punctuation, numbers
def clean_text(text):
    # lowercase
    text = str(text).lower()
    # remove urls
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"www\S+", "", text)

    # remove html tags
    text = re.sub(r"<.*?>", "", text)
    # Remove numbers
    text = re.sub(r"\d+", "", text)
    # Remove punctuation
    text = text.translate(
        str.maketrans("", "", string.punctuation)
    )
    # Replace multiple spaces with single space
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [10]:
# STEP 6 — CLEAN ALL NEWS ARTICLES
df["text"] = df["text"].apply(clean_text)

In [11]:
# STEP 7 — DEFINE FEATURES AND LABELS
# X = News Text
# y = Fake/Real Label
X = df["text"]
y = df["label"]

In [12]:
# STEP 8 — TRAIN TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Training Samples:", len(X_train))
print("Testing Samples:", len(X_test))

Training Samples: 57676
Testing Samples: 14419


In [13]:
# STEP 9 — TOKENIZATION
# Convert text into integer sequences
VOCAB_SIZE = 15000

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

print("Tokenizer Ready")

Tokenizer Ready


In [14]:
# STEP 10 — TEXT TO SEQUENCES
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

In [16]:
# STEP 11 — PADDING SEQUENCES
MAX_LEN = 200

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding="post", truncating="post")

X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN, padding="post", truncating="post")

print(X_train_pad.shape)
print(X_test_pad.shape)

(57676, 200)
(14419, 200)


In [17]:
# STEP 12 — BUILD BiLSTM MODEL
model = Sequential([

    Embedding(input_dim=VOCAB_SIZE, output_dim=64, input_length=MAX_LEN),

    Bidirectional(LSTM(32, dropout=0.2, recurrent_dropout=0.2)),

    Dense(32, activation="relu"),
    Dropout(0.3),

    Dense(1, activation="sigmoid")])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [18]:
# STEP 12 — BUILD BiLSTM MODEL
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

model.compile(optimizer=optimizer, loss="binary_crossentropy", metrics=["accuracy"])

print("Model Compiled")

Model Compiled


In [19]:
# STEP 12 — BUILD BiLSTM MODEL
model.build(input_shape=(None, MAX_LEN))

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 200, 64)        │       960,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 64)             │        24,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 986,945 (3.76 MB)

 Trainable params: 986,945 (3.76 MB)

 Non-trainable params: 0 (0.00 B)

In [20]:
# STEP 15 — EARLY STOPPING
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

In [21]:
# STEP 16 — TRAIN MODEL
history = model.fit(X_train_pad, y_train, validation_split=0.1,epochs=5, batch_size=128, callbacks=[early_stop])

Epoch 1/5
406/406 ━━━━━━━━━━━━━━━━━━━━ 519s 1s/step - accuracy: 0.9087 - loss: 0.2277 - val_accuracy: 0.9541 - val_loss: 0.1226
Epoch 2/5
406/406 ━━━━━━━━━━━━━━━━━━━━ 498s 1s/step - accuracy: 0.9680 - loss: 0.0929 - val_accuracy: 0.9629 - val_loss: 0.1097
Epoch 3/5
406/406 ━━━━━━━━━━━━━━━━━━━━ 500s 1s/step - accuracy: 0.9806 - loss: 0.0580 - val_accuracy: 0.9560 - val_loss: 0.1237
Epoch 4/5
406/406 ━━━━━━━━━━━━━━━━━━━━ 506s 1s/step - accuracy: 0.9793 - loss: 0.0588 - val_accuracy: 0.9626 - val_loss: 0.1196


In [22]:
# STEP 17 — EVALUATE MODEL
loss, accuracy = model.evaluate(X_test_pad, y_test)

print("Accuracy:", accuracy)

451/451 ━━━━━━━━━━━━━━━━━━━━ 111s 247ms/step - accuracy: 0.9636 - loss: 0.0976
Accuracy: 0.9635897278785706


In [23]:
# STEP 18 — GENERATE PREDICTIONS
y_pred_prob = model.predict(X_test_pad)

y_pred = (y_pred_prob > 0.5).astype(int)
print("Predictions Completed")

451/451 ━━━━━━━━━━━━━━━━━━━━ 112s 247ms/step
Predictions Completed


In [24]:
# STEP 19 — CLASSIFICATION REPORT
# Precision, Recall, F1 Score

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))


Classification Report:

              precision    recall  f1-score   support

           0       0.96      0.96      0.96      7006
           1       0.97      0.96      0.96      7413

    accuracy                           0.96     14419
   macro avg       0.96      0.96      0.96     14419
weighted avg       0.96      0.96      0.96     14419



In [25]:
# STEP 20 — CONFUSION MATRIX

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))


Confusion Matrix:

[[6749  257]
 [ 268 7145]]


In [26]:
# STEP 21 - NEWS PREDICTION FUNCTION
def predict_news(news):

    news = clean_text(news)
    seq = tokenizer.texts_to_sequences([news])

    padded = pad_sequences(seq, maxlen=MAX_LEN, padding="post", truncating="post")
    prob = model.predict(padded, verbose=0)[0][0]

    print(f"Fake Probability: {prob:.4f}")

    if prob >= 0.7:
        print("🔴 FAKE")

    elif prob <= 0.3:
        print("🟢 REAL")

    else:
        print("🟡 UNCERTAIN")

In [27]:
fake = "No comment is expected from Barack Obama Members of the #FYF911 or #FukYoFlag and #BlackLivesMatter movements called for the lynching and hanging of white people and cops. They encouraged others on a radio show Tuesday night to  turn the tide  and kill white people and cops to send a message about the killing of black people in America.One of the F***YoFlag organizers is called  Sunshine.  She has a radio blog show hosted from Texas called,  Sunshine s F***ing Opinion Radio Show. A snapshot of her #FYF911 @LOLatWhiteFear Twitter page at 9:53 p.m. shows that she was urging supporters to  Call now!! #fyf911 tonight we continue to dismantle the illusion of white Below is a SNAPSHOT Twitter Radio Call Invite   #FYF911The radio show aired at 10:00 p.m. eastern standard time.During the show, callers clearly call for  lynching  and  killing  of white people.A 2:39 minute clip from the radio show can be heard here. It was provided to Breitbart Texas by someone who would like to be referred to as  Hannibal.  He has already received death threats as a result of interrupting #FYF911 conference calls.An unidentified black man said  when those mother f**kers are by themselves, that s when when we should start f***ing them up. Like they do us, when a bunch of them ni**ers takin  one of us out, that s how we should roll up.  He said,  Cause we already roll up in gangs anyway. There should be six or seven black mother f**ckers, see that white person, and then lynch their ass. Let s turn the tables. They conspired that if  cops started losing people,  then  there will be a state of emergency. He speculated that one of two things would happen,  a big-ass [R s?????] war,  or  ni**ers, they are going to start backin  up. We are already getting killed out here so what the f**k we got to lose? Sunshine could be heard saying,  Yep, that s true. That s so f**king true. He said,  We need to turn the tables on them. Our kids are getting shot out here. Somebody needs to become a sacrifice on their side.He said,  Everybody ain t down for that s**t, or whatever, but like I say, everybody has a different position of war.  He continued,  Because they don t give a f**k anyway.  He said again,  We might as well utilized them for that s**t and turn the tables on these n**ers. He said, that way  we can start lookin  like we ain t havin  that many casualties, and there can be more causalities on their side instead of ours. They are out their killing black people, black lives don t matter, that s what those mother f**kers   so we got to make it matter to them. Find a mother f**ker that is alone. Snap his ass, and then f***in hang him from a damn tree. Take a picture of it and then send it to the mother f**kers. We  just need one example,  and  then people will start watchin .  This will turn the tables on s**t, he said. He said this will start  a trickle-down effect.  He said that when one white person is hung and then they are just  flat-hanging,  that will start the  trickle-down effect.  He continued,  Black people are good at starting trends. He said that was how  to get the upper-hand. Another black man spoke up saying they needed to kill  cops that are killing us. The first black male said,  That will be the best method right there. Breitbart Texas previously reported how Sunshine was upset when  racist white people  infiltrated and disrupted one of her conference calls. She subsequently released the phone number of one of the infiltrators. The veteran immediately started receiving threatening calls.One of the #F***YoFlag movement supporters allegedly told a veteran who infiltrated their publicly posted conference call,  We are going to rape and gut your pregnant wife, and your f***ing piece of sh*t unborn creature will be hung from a tree. Breitbart Texas previously encountered Sunshine at a Sandra Bland protest at the Waller County Jail in Texas, where she said all white people should be killed. She told journalists and photographers,  You see this nappy-ass hair on my head?   That means I am one of those more militant Negroes.  She said she was at the protest because  these redneck mother-f**kers murdered Sandra Bland because she had nappy hair like me. #FYF911 black radicals say they will be holding the  imperial powers  that are actually responsible for the terrorist attacks on September 11th accountable on that day, as reported by Breitbart Texas. There are several websites and Twitter handles for the movement. Palmetto Star  describes himself as one of the head organizers. He said in a YouTube video that supporters will be burning their symbols of  the illusion of their superiority,  their  false white supremacy,  like the American flag, the British flag, police uniforms, and Ku Klux Klan hoods.Sierra McGrone or  Nocturnus Libertus  posted,  you too can help a young Afrikan clean their a** with the rag of oppression.  She posted two photos, one that appears to be herself, and a photo of a black man, wiping their naked butts with the American flag.For entire story: Breitbart News"
predict_news(fake)

Fake Probability: 0.9972
🔴 FAKE


In [28]:

real = "BRUSSELS (Reuters) - British Prime Minister Theresa May s offer of  settled status  for EU residents is flawed and will leave them with fewer rights after Brexit, the European"
predict_news(real)

Fake Probability: 0.0037
🟢 REAL


In [31]:
# 22 Model Save
model.save("fake_news_model.keras")

print("Model Saved")

Model Saved
